# Read Public Test Cases

This notebook loads every case folder inside `PublicTestCases`. If pandas is installed, tables are returned as DataFrames; otherwise they are returned as lists of dictionaries.

In [1]:
import csv
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

if pd is not None:
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 120)

In [2]:
DATA_DIR = Path("PublicTestCases")

CSV_SCHEMAS = {
    "warehouse": ["x", "y"],
    "types_of_bays": [
        "type_id",
        "width",
        "depth",
        "height",
        "min_spacing",
        "capacity",
        "cost",
    ],
    "obstacles": ["x", "y", "width", "height"],
    "ceiling": ["x", "height"],
}


def parse_value(value: str) -> int | float | str:
    value = value.strip()
    try:
        return int(value)
    except ValueError:
        try:
            return float(value)
        except ValueError:
            return value


def read_csv_rows(csv_path: Path, columns: list[str]):
    with csv_path.open(newline="") as file:
        reader = csv.reader(file, skipinitialspace=True)
        rows = [
            {column: parse_value(value) for column, value in zip(columns, row)}
            for row in reader
            if row
        ]

    return pd.DataFrame(rows, columns=columns) if pd is not None else rows


def read_case(case_dir: Path) -> dict[str, object]:
    """Read one PublicTestCases/Case* directory."""
    tables = {}
    for table_name, columns in CSV_SCHEMAS.items():
        csv_path = case_dir / f"{table_name}.csv"
        if not csv_path.exists():
            raise FileNotFoundError(f"Missing expected file: {csv_path}")

        tables[table_name] = read_csv_rows(csv_path, columns)
    return tables


def load_public_test_cases(data_dir: Path = DATA_DIR) -> dict[str, dict[str, object]]:
    """Load all Case* folders under PublicTestCases."""
    if not data_dir.exists():
        raise FileNotFoundError(f"Could not find data folder: {data_dir.resolve()}")

    case_dirs = sorted(path for path in data_dir.iterdir() if path.is_dir() and path.name.startswith("Case"))
    return {case_dir.name: read_case(case_dir) for case_dir in case_dirs}


cases = load_public_test_cases()
list(cases)

['Case0', 'Case1', 'Case2', 'Case3']

In [3]:
summary_rows = [
    {
        "case": case_name,
        **{table_name: len(table) for table_name, table in tables.items()},
    }
    for case_name, tables in cases.items()
]

summary = pd.DataFrame(summary_rows) if pd is not None else summary_rows
summary

[{'case': 'Case0',
  'warehouse': 6,
  'types_of_bays': 6,
  'obstacles': 3,
  'ceiling': 3},
 {'case': 'Case1',
  'warehouse': 4,
  'types_of_bays': 16,
  'obstacles': 0,
  'ceiling': 2},
 {'case': 'Case2',
  'warehouse': 4,
  'types_of_bays': 16,
  'obstacles': 1,
  'ceiling': 2},
 {'case': 'Case3',
  'warehouse': 20,
  'types_of_bays': 16,
  'obstacles': 1,
  'ceiling': 3}]

In [4]:
# Pick a case to inspect.
case_name = "Case0"
case = cases[case_name]

case.keys()

dict_keys(['warehouse', 'types_of_bays', 'obstacles', 'ceiling'])

In [5]:
case["warehouse"]

[{'x': 0, 'y': 0},
 {'x': 10000, 'y': 0},
 {'x': 10000, 'y': 3000},
 {'x': 3000, 'y': 3000},
 {'x': 3000, 'y': 10000},
 {'x': 0, 'y': 10000}]

In [6]:
case["types_of_bays"]

[{'type_id': 0,
  'width': 800,
  'depth': 1200,
  'height': 2800,
  'min_spacing': 200,
  'capacity': 4,
  'cost': 2000},
 {'type_id': 1,
  'width': 1600,
  'depth': 1200,
  'height': 2800,
  'min_spacing': 200,
  'capacity': 8,
  'cost': 2500},
 {'type_id': 2,
  'width': 2400,
  'depth': 1200,
  'height': 2800,
  'min_spacing': 200,
  'capacity': 12,
  'cost': 2800},
 {'type_id': 3,
  'width': 800,
  'depth': 1000,
  'height': 1800,
  'min_spacing': 150,
  'capacity': 3,
  'cost': 1800},
 {'type_id': 4,
  'width': 1600,
  'depth': 1000,
  'height': 1800,
  'min_spacing': 150,
  'capacity': 6,
  'cost': 2300},
 {'type_id': 5,
  'width': 2400,
  'depth': 1000,
  'height': 1800,
  'min_spacing': 150,
  'capacity': 9,
  'cost': 2600}]

In [ ]:
case["obstacles"]

[{'x': 750, 'y': 750, 'width': 750, 'height': 750},
 {'x': 8000, 'y': 2500, 'width': 1500, 'height': 300},
 {'x': 1500, 'y': 4200, 'width': 200, 'height': 4600}]

In [ ]:
case["ceiling"]

[{'x': 0, 'height': 3000},
 {'x': 3000, 'height': 2000},
 {'x': 6000, 'height': 3000}]

The loaded data is available in the nested dictionary `cases`:

- `cases["Case0"]["warehouse"]`
- `cases["Case0"]["types_of_bays"]`
- `cases["Case0"]["obstacles"]`
- `cases["Case0"]["ceiling"]`